# MoodMap Voice Emotion Analyzer — Training Notebook

Fine-tunes **HuBERT-base** on a combined 4-dataset blend for 7-class speech emotion classification.

| Index | Emotion | Datasets with this label |
|-------|---------|-------------------------|
| 0 | happy | RAVDESS, CREMA-D, TESS, SAVEE |
| 1 | sad | RAVDESS, CREMA-D, TESS, SAVEE |
| 2 | angry | RAVDESS, CREMA-D, TESS, SAVEE |
| 3 | fearful | RAVDESS, CREMA-D, TESS, SAVEE |
| 4 | disgust | RAVDESS, CREMA-D, TESS, SAVEE |
| 5 | surprised | RAVDESS, TESS, SAVEE |
| 6 | neutral | RAVDESS, CREMA-D, TESS, SAVEE |

Why HuBERT over Wav2Vec2? Research shows HuBERT consistently outperforms Wav2Vec2 on speech emotion tasks.
Why 7 classes instead of 9? Speech datasets don't contain "love" or "optimism" — those are text-only emotions.
The backend maps these 7 voice emotions to the 9-emotion system (love=0, optimism=0 from voice).

**Combined dataset: ~12,162 samples** (vs 1,440 in previous RAVDESS-only approach)

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn accelerate librosa soundfile kaggle

## 1. Download All Datasets

In [ ]:
import os
import json
from google.colab import userdata

# Use Colab Secrets — never hardcode API keys
kaggle_creds = {
    "username": userdata.get("KAGGLE_USERNAME"),
    "key": userdata.get("KAGGLE_KEY"),
}

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(kaggle_creds, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

print("Downloading datasets from Kaggle...")
!kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio -q
!kaggle datasets download -d ejlok1/cremad -q
!kaggle datasets download -d ejlok1/toronto-emotional-speech-set-tess -q
!kaggle datasets download -d ejlok1/surrey-audiovisual-expressed-emotion-savee -q

print("Extracting...")
!unzip -q -o ravdess-emotional-speech-audio.zip -d ./ravdess_data
!unzip -q -o cremad.zip -d ./cremad_data
!unzip -q -o toronto-emotional-speech-set-tess.zip -d ./tess_data
!unzip -q -o surrey-audiovisual-expressed-emotion-savee.zip -d ./savee_data
print("All datasets ready!")

## 2. Build Unified Dataset

Map all 4 datasets to a common 7-emotion label set.

In [ ]:
import glob
import pandas as pd
import librosa
import numpy as np
from datasets import Dataset, DatasetDict
from collections import Counter

VOICE_EMOTIONS = ["happy", "sad", "angry", "fearful", "disgust", "surprised", "neutral"]
NUM_LABELS = len(VOICE_EMOTIONS)

data = []

# ── RAVDESS ────────────────────────────────────────────────
# Filename format: 03-01-{emotion}-{intensity}-{statement}-{rep}-{actor}.wav
# Emotions: 01=neutral, 02=calm, 03=happy, 04=sad, 05=angry, 06=fearful, 07=disgust, 08=surprised
RAVDESS_MAP = {"01": 6, "02": 6, "03": 0, "04": 1, "05": 2, "06": 3, "07": 4, "08": 5}

for f in glob.glob("./ravdess_data/**/*.wav", recursive=True):
    emotion_code = os.path.basename(f).split("-")[2]
    if emotion_code in RAVDESS_MAP:
        data.append({"file_path": f, "label": RAVDESS_MAP[emotion_code], "source": "ravdess"})

print(f"RAVDESS: {sum(1 for d in data if d['source']=='ravdess')} files")

# ── CREMA-D ────────────────────────────────────────────────
# Filename format: {actor}_{sentence}_{emotion}_{intensity}.wav
# Emotions: ANG, DIS, FEA, HAP, NEU, SAD
CREMAD_MAP = {"ANG": 2, "DIS": 4, "FEA": 3, "HAP": 0, "NEU": 6, "SAD": 1}

cremad_before = len(data)
for f in glob.glob("./cremad_data/**/*.wav", recursive=True):
    parts = os.path.basename(f).replace(".wav", "").split("_")
    if len(parts) >= 3:
        emotion_code = parts[2]
        if emotion_code in CREMAD_MAP:
            data.append({"file_path": f, "label": CREMAD_MAP[emotion_code], "source": "cremad"})

print(f"CREMA-D: {sum(1 for d in data if d['source']=='cremad')} files")

# ── TESS ───────────────────────────────────────────────────
# Folder structure: TESS/{speaker}_{emotion}/*.wav
# Emotions: angry, disgust, fear, happy, ps (pleasant surprise), sad, neutral
TESS_MAP = {"angry": 2, "disgust": 4, "fear": 3, "happy": 0, "ps": 5, "sad": 1, "neutral": 6}

for f in glob.glob("./tess_data/**/*.wav", recursive=True):
    folder_name = os.path.basename(os.path.dirname(f)).lower()
    for emo_key, label_idx in TESS_MAP.items():
        if emo_key in folder_name:
            data.append({"file_path": f, "label": label_idx, "source": "tess"})
            break

print(f"TESS: {sum(1 for d in data if d['source']=='tess')} files")

# ── SAVEE ──────────────────────────────────────────────────
# Filename format: {actor}_{emotion}{number}.wav
# Emotion codes: a=angry, d=disgust, f=fear, h=happy, n=neutral, sa=sad, su=surprise
SAVEE_MAP = {"a": 2, "d": 4, "f": 3, "h": 0, "n": 6, "sa": 1, "su": 5}

for f in glob.glob("./savee_data/**/*.wav", recursive=True):
    basename = os.path.basename(f).replace(".wav", "").lower()
    # Extract emotion code (letters before the digits at end)
    parts = basename.split("_")
    if len(parts) >= 2:
        emo_part = "".join(c for c in parts[-1] if c.isalpha())
        if emo_part in SAVEE_MAP:
            data.append({"file_path": f, "label": SAVEE_MAP[emo_part], "source": "savee"})

print(f"SAVEE: {sum(1 for d in data if d['source']=='savee')} files")

df = pd.DataFrame(data)
print(f"\nTotal samples: {len(df):,}")
print(f"\nLabel distribution:")
for i, name in enumerate(VOICE_EMOTIONS):
    count = len(df[df["label"] == i])
    print(f"  {name:10s}: {count:>5}")

In [ ]:
from transformers import AutoFeatureExtractor

feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/hubert-base-ls960")
TARGET_SR = 16000
MAX_DURATION_SEC = 6.0
MAX_LENGTH = int(TARGET_SR * MAX_DURATION_SEC)

def load_and_process(example):
    audio_array, sr = librosa.load(example["file_path"], sr=TARGET_SR)
    # Truncate or pad to fixed length for consistent batching
    if len(audio_array) > MAX_LENGTH:
        audio_array = audio_array[:MAX_LENGTH]
    inputs = feature_extractor(
        audio_array, sampling_rate=TARGET_SR, return_tensors="np",
        padding="max_length", max_length=MAX_LENGTH, truncation=True,
    )
    example["input_values"] = inputs.input_values[0]
    return example

print("Processing all audio files (this takes a few minutes)...")
full_dataset = Dataset.from_pandas(df)
full_dataset = full_dataset.map(load_and_process, remove_columns=["file_path", "source"])

# Stratified split: 80% train, 10% val, 10% test
split1 = full_dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
split2 = split1["test"].train_test_split(test_size=0.5, seed=42, stratify_by_column="label")

dataset = DatasetDict({
    "train": split1["train"],
    "validation": split2["train"],
    "test": split2["test"],
})

print(f"\nDataset splits:")
print(f"  Train:      {len(dataset['train']):,}")
print(f"  Validation: {len(dataset['validation']):,}")
print(f"  Test:       {len(dataset['test']):,}")

## 3. Model Setup

Fine-tune HuBERT-base with frozen feature encoder (only train the classifier head + transformer layers).

In [ ]:
import torch
from transformers import (
    HubertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

model = HubertForSequenceClassification.from_pretrained(
    "facebook/hubert-base-ls960",
    num_labels=NUM_LABELS,
    problem_type="single_label_classification",
)

# Freeze the CNN feature extractor — only fine-tune the transformer + classifier
model.freeze_feature_encoder()

# Label mapping for the model config
model.config.label2id = {name: i for i, name in enumerate(VOICE_EMOTIONS)}
model.config.id2label = {i: name for i, name in enumerate(VOICE_EMOTIONS)}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_weighted": f1_score(labels, predictions, average="weighted"),
        "f1_macro": f1_score(labels, predictions, average="macro"),
    }

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Union

@dataclass
class AudioDataCollator:
    feature_extractor: any

    def __call__(self, features: List[Dict[str, Union[List[float], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        labels = [f["label"] for f in features]

        batch = self.feature_extractor.pad(
            input_features, padding=True, return_tensors="pt",
        )
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

training_args = TrainingArguments(
    output_dir="./moodmap-hubert-results",

    learning_rate=1e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,

    logging_steps=50,
    report_to="none",
    remove_unused_columns=False,

    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=feature_extractor,
    data_collator=AudioDataCollator(feature_extractor=feature_extractor),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Starting training...")
trainer.train()

## 4. Evaluation — Per-Class Metrics

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(dataset["test"])
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

print("Per-Class Classification Report:")
print("=" * 60)
print(classification_report(
    true_labels, pred_labels,
    target_names=VOICE_EMOTIONS,
    digits=3,
))

print("\nConfusion Matrix:")
cm = confusion_matrix(true_labels, pred_labels)
print(f"{'':12s}", "  ".join(f"{e[:4]:>5s}" for e in VOICE_EMOTIONS))
for i, row in enumerate(cm):
    print(f"{VOICE_EMOTIONS[i]:12s}", "  ".join(f"{v:>5d}" for v in row))

## 5. Sanity Check — Voice Emotion to Valence Mapping

Verify that the voice emotion scores produce sensible valence values
when mapped through the same formula the backend uses.

In [ ]:
import torch.nn.functional as F

# The 9-category mapping used in the backend
VOICE_TO_9 = {
    "happy": "joy", "sad": "sadness", "angry": "anger",
    "fearful": "fear", "disgust": "disgust", "surprised": "surprise", "neutral": "neutral",
}
NINE_EMOTIONS = ["joy", "love", "optimism", "sadness", "anger", "fear", "disgust", "surprise", "neutral"]

def voice_to_valence(logits_tensor):
    """Same logic the backend will use."""
    probs = F.softmax(logits_tensor, dim=-1).squeeze().numpy()
    scores_7 = {VOICE_EMOTIONS[i]: float(probs[i]) for i in range(NUM_LABELS)}

    # Map to 9-category system
    scores_9 = {e: 0.0 for e in NINE_EMOTIONS}
    for v_emo, score in scores_7.items():
        scores_9[VOICE_TO_9[v_emo]] = score

    pos_avg = (scores_9["joy"] + scores_9["love"] + scores_9["optimism"]) / 3.0
    neg_avg = (scores_9["sadness"] + scores_9["anger"] + scores_9["fear"] + scores_9["disgust"]) / 4.0
    neutral = scores_9["neutral"]
    valence = (pos_avg - neg_avg) * (1.0 - 0.6 * neutral)
    return max(-1.0, min(1.0, valence)), scores_7

# Test on a few samples from the test set
print("Voice Emotion → Valence Sanity Check")
print("=" * 70)
for i in range(min(10, len(dataset["test"]))):
    sample = dataset["test"][i]
    inputs = feature_extractor(
        sample["input_values"], sampling_rate=TARGET_SR, return_tensors="pt", padding=True,
    )
    with torch.no_grad():
        logits = model(**inputs).logits

    true_emo = VOICE_EMOTIONS[sample["label"]]
    pred_emo = VOICE_EMOTIONS[logits.argmax(-1).item()]
    valence, scores = voice_to_valence(logits)

    print(f"  True: {true_emo:10s} | Pred: {pred_emo:10s} | Valence: {valence:+.3f}")

## 6. Upload to Hugging Face

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token, add_to_git_credential=True)

repo_name = "MNauman13/moodmap-hubert"

print(f"Uploading model to {repo_name}...")
model.push_to_hub(repo_name)
feature_extractor.push_to_hub(repo_name)
print("Upload complete!")